# 04 · Planning GPU memory: weights, KV cache and concurrency

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/vllm/04_memory_planner.ipynb) &nbsp; 📖 Concept deck: [Running and tuning vLLM](https://nilesh-salpe.github.io/visual-ai/vllm/08-running-and-tuning/) &nbsp; ⏱️ Suggested time: **25 min**

**You will build:** how `gpu_memory_utilization` splits a GPU into weights, activation workspace and KV cache, the KV bytes per token of a model, and the resulting maximum number of concurrent tokens and requests.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


GiB = 2 ** 30
MODELS = {  # layers, KV heads, head dim, parameters (billions)
    "Llama-3.1-8B":  dict(L=32, kv=8, d=128, P=8.03),
    "Llama-3.1-70B": dict(L=80, kv=8, d=128, P=70.6),
    "Qwen2.5-7B":    dict(L=28, kv=4, d=128, P=7.62),
}

## Task 1: KV bytes per token

Each token stores a key and a value vector per layer per KV head: `2 × L × kv_heads × head_dim × bytes`. Return it for a model dict and a dtype size (2 for BF16, 1 for an FP8 KV cache).

In [ ]:
def kv_bytes_per_token(m, dtype_bytes=2):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_kv_bytes_per_token(f):
    assert f(MODELS["Llama-3.1-8B"]) == 131072, "Llama-3.1-8B: 128 KiB per token in BF16"
    assert f(MODELS["Llama-3.1-8B"], 1) == 65536

check("kv_bytes_per_token", _test_kv_bytes_per_token, kv_bytes_per_token)

## Task 2: How many tokens fit?

vLLM uses `gpu_memory_utilization × GPU memory` in total (default **0.92** in v0.31.0). From that it subtracts the weights (P × bytes per weight) and the activation peak measured while profiling, and gives the rest to the KV cache.
Return `(kv_gib, max_tokens, max_requests)` for a context of `ctx` tokens per request (max_requests = floor(max_tokens / ctx)).

In [ ]:
def kv_capacity(m, gpu_gib=80, util=0.92, weight_bytes=2, act_gib=4, kv_dtype_bytes=2, ctx=8192):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_kv_capacity(f):
    kv, tok, req = f(MODELS["Llama-3.1-8B"])
    assert abs(kv - (0.92 * 80 - 8.03e9 * 2 / GiB - 4)) < 1e-6
    assert tok == int(kv * GiB // 131072) and req == tok // 8192
    assert f(MODELS["Llama-3.1-8B"], kv_dtype_bytes=1)[1] == 2 * tok or abs(f(MODELS["Llama-3.1-8B"], kv_dtype_bytes=1)[1] - 2 * tok) <= 1, "FP8 KV doubles capacity"
    assert f(MODELS["Llama-3.1-70B"])[0] < 0, "70B in BF16 does not fit on one 80 GB GPU"

check("kv_capacity", _test_kv_capacity, kv_capacity)

## Task 3: Tensor parallelism frees KV space

With tensor parallelism `tp`, each GPU holds 1/tp of the weights and of the KV heads (so 1/tp of the KV bytes per token). Return the total number of KV tokens the `tp` GPUs together can hold for the model (same per-GPU budget as above).

In [ ]:
def tp_capacity(m, tp, gpu_gib=80, util=0.92, weight_bytes=2, act_gib=4, kv_dtype_bytes=2):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_tp_capacity(f):
    m = MODELS["Llama-3.1-70B"]
    assert f(m, 1) == 0 and f(m, 2) > 0, "70B fits with TP = 2"
    assert f(m, 4) > 2 * f(m, 2), "doubling TP more than doubles KV capacity: weights per GPU shrink too"

check("tp_capacity", _test_tp_capacity, tp_capacity)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: KV bytes per token
def kv_bytes_per_token_solution(m, dtype_bytes=2):
    return 2 * m["L"] * m["kv"] * m["d"] * dtype_bytes

check("kv_bytes_per_token (reference)", _test_kv_bytes_per_token, kv_bytes_per_token_solution)

In [ ]:
# @title Solution for Task 2: How many tokens fit?
def kv_capacity_solution(m, gpu_gib=80, util=0.92, weight_bytes=2, act_gib=4, kv_dtype_bytes=2, ctx=8192):
    kv_gib = util * gpu_gib - m["P"] * 1e9 * weight_bytes / GiB - act_gib
    tokens = int(kv_gib * GiB // kv_bytes_per_token_solution(m, kv_dtype_bytes)) if kv_gib > 0 else 0
    return kv_gib, tokens, tokens // ctx

check("kv_capacity (reference)", _test_kv_capacity, kv_capacity_solution)

In [ ]:
# @title Solution for Task 3: Tensor parallelism frees KV space
def tp_capacity_solution(m, tp, gpu_gib=80, util=0.92, weight_bytes=2, act_gib=4, kv_dtype_bytes=2):
    per_gpu = util * gpu_gib - m["P"] * 1e9 * weight_bytes / GiB / tp - act_gib
    if per_gpu <= 0: return 0
    return int(per_gpu * GiB // (kv_bytes_per_token_solution(m, kv_dtype_bytes) / tp))

check("tp_capacity (reference)", _test_tp_capacity, tp_capacity_solution)

---
## Experiments
**A planning table.** KV capacity and concurrent 8K-token requests for each model on an 80 GB GPU, in BF16 and with an FP8 KV cache (`--kv-cache-dtype fp8`).

In [ ]:
for name, m in MODELS.items():
    for tp in [1, 2, 4]:
        bf, f8 = tp_capacity_solution(m, tp), tp_capacity_solution(m, tp, kv_dtype_bytes=1)
        if bf or f8: print(f"{name:14s} TP={tp}: {bf:9,d} KV tokens BF16 ({bf // 8192:4d} × 8K requests) | FP8 KV {f8:9,d} ({f8 // 8192:4d})")

---
## Takeaways
* KV bytes per token = 2 × layers × KV heads × head dim × bytes; GQA (few KV heads) and FP8 KV caches shrink it.
* vLLM reserves `gpu_memory_utilization` (0.92 by default in v0.31.0) of GPU memory, subtracts weights and the profiled activation peak, and pages the rest as KV blocks.
* Tensor parallelism splits both weights and KV heads, so KV capacity grows faster than the GPU count.